# M2b · Cleaning-rule exploration

**Goal:** for each of C1–C7, count how many rows would be **rejected**, **clamped**, or **nullified** if we applied the rule to staging today.

No writes. The cleaning rules live in `config/cleaning_rules.yaml` and the engine in `src/accent_fleet/cleaning/rules.py`.

**Exit criterion:** you can state rejection % per rule per source table.


In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = accent_app@host.docker.internal:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs

In [2]:
from accent_fleet.config import load_cleaning_rules
rules = load_cleaning_rules()
import json
print(json.dumps(rules, indent=2, default=str)[:1500], '...')


{
  "version": 1,
  "updated": "2026-04-20",
  "rules": [
    {
      "id": "C1",
      "name": "temporal_epoch_error_filter",
      "severity": "critical",
      "targets": [
        "path",
        "stop",
        "rep_overspeed",
        "notification",
        "rep_activity_daily"
      ],
      "time_column": {
        "path": "begin_path_time",
        "stop": "stop_start",
        "rep_overspeed": "begin_path_time",
        "notification": "created_at",
        "rep_activity_daily": "activity_start_time"
      },
      "condition": "{time_column} >= '2019-10-01'::timestamp",
      "rationale": "GPS devices without a synchronized clock default to Unix epoch (1970) or GPS epoch (1980). Earliest valid tenant data is 2019-10-01.\n",
      "action": "reject",
      "enabled": true
    },
    {
      "id": "C2",
      "name": "trip_duration_positive",
      "severity": "critical",
      "targets": [
        "path"
      ],
      "condition": "path_duration > 0",
      "rationale": "14

## 3. Execute — count would-be rejects per rule directly in SQL

In [3]:
# We translate each rule's SQL condition into a COUNT probe. The YAML condition
# is TRUE for valid rows, so affected rows are the rows that do NOT satisfy it.
import pandas as pd
probes = []
for r in rules.get('rules', []):
    rid = r['id']; action = r['action']
    for tbl in r.get('targets', []):
        cond = r.get('condition', 'TRUE')
        if '{time_column}' in cond:
            cond = cond.format(time_column=r.get('time_column', {}).get(tbl, 'event_time'))
        q = text(f"SELECT COUNT(*) FROM staging.{tbl} WHERE NOT ({cond})")
        try:
            with get_engine().connect() as conn:
                n = conn.execute(q).scalar_one()
            probes.append({'rule': rid, 'action': action, 'table': tbl, 'matched': n})
        except Exception as exc:
            probes.append({'rule': rid, 'action': action, 'table': tbl, 'matched': f'ERR: {exc}'})
pd.DataFrame(probes)


,rule,action,table,matched
0,C1,reject,path,ERR: (psycopg.errors.ConnectionTimeout) connec...
1,C1,reject,stop,ERR: (psycopg.errors.ConnectionTimeout) connec...
2,C1,reject,rep_overspeed,ERR: (psycopg.errors.ConnectionTimeout) connec...
3,C1,reject,notification,ERR: (psycopg.errors.ConnectionTimeout) connec...
4,C1,reject,rep_activity_daily,ERR: (psycopg.errors.ConnectionTimeout) connec...
5,C2,reject,path,ERR: (psycopg.errors.ConnectionTimeout) connec...
6,C3,reject,path,ERR: (psycopg.errors.ConnectionTimeout) connec...
7,C4,nullify,path,ERR: (psycopg.errors.ConnectionTimeout) connec...
8,C5,clamp,path,ERR: (psycopg.errors.ConnectionTimeout) connec...
9,C5,clamp,rep_overspeed,ERR: (psycopg.errors.ConnectionTimeout) connec...


## 4. Inspect

In [4]:
# Compare to total rows per table so you can talk in percentages.
target_tables = sorted({tbl for r in rules.get('rules', []) for tbl in r.get('targets', [])})
with get_engine().connect() as conn:
    totals = {tbl: conn.execute(text(f"SELECT COUNT(*) FROM staging.{tbl}")).scalar_one()
              for tbl in target_tables}
import pandas as pd
df = pd.DataFrame(probes)
df['total'] = df['table'].map(totals)
df['pct'] = (df['matched'].where(df['matched'].apply(lambda v: isinstance(v, (int,float))), 0) / df['total'] * 100).round(3)
df


OperationalError: (psycopg.errors.ConnectionTimeout) connection timeout expired
(Background on this error at: https://sqlalche.me/e/20/e3q8)